# Simple Snowflake ETL Workflow
## D382_ETL — MovieLens movies.csv to analytics tables

Use **`C:\data\movielens\ml-latest-small\movies\movies.csv`** as the source file. Create an internal stage, upload the file, load a raw table, clean movie records, split genre lists, and build a genre summary.

This Markdown notebook contains copyable commands. Run SQL in Snowflake; run `PUT` through a local Snowflake client, or upload with Snowsight. Local file checks run in PowerShell.

**Schema assumption:** standard MovieLens `movieId,title,genres`. The requested local file was not accessible from the authoring workspace; confirm its header in step 2. No fixed dataset size is assumed. Movie ratings are in a separate MovieLens file and are outside this workflow.

Requires an authorized Snowflake role and warehouse access. SQL has not been executed against a live account.

## 1. Understand the complete workflow

```text
C:/data/movielens/ml-latest-small/movies/movies.csv
       | upload
       v
@MOVIES_ETL_DB.RAW.MOVIES_STAGE/movielens_001/movies.csv
       | COPY INTO
       v
RAW.MOVIELENS_MOVIES_RAW (three source fields + file metadata)
       | validate and clean
       v
CURATED.MOVIELENS_CLASSIFIED
       |                         |
       v                         v
RAW.MOVIELENS_REJECTED    CURATED.MOVIELENS_MOVIES
                                 | split pipe-separated genres
                                 v
                         CURATED.MOVIELENS_MOVIE_GENRES
                                 | aggregate
                                 v
                         ANALYTICS.MOVIELENS_GENRE_SUMMARY
```

This is an ELT implementation of the ETL workflow: extract/export to CSV, load source values, then transform in Snowflake. Uploading stores a file; `COPY INTO` inserts rows into a table.

MovieLens-specific table names keep this workflow separate from the earlier five-column teaching dataset.

## 2. Verify the source file and its schema
 
The expected header is:

```csv
movieId,title,genres
```

Illustrative standard MovieLens records (use your actual file, not this excerpt):

```csv
movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
```
 

## 3. Create the database, schemas, and warehouse

**Snowflake SQL.** Use your authorized role. If your role cannot create databases or warehouses, use administrator-assigned resources and replace the names consistently throughout the notebook.

```sql
CREATE DATABASE IF NOT EXISTS MOVIES_ETL_DB;
USE DATABASE MOVIES_ETL_DB;

CREATE SCHEMA IF NOT EXISTS RAW;
CREATE SCHEMA IF NOT EXISTS CURATED;
CREATE SCHEMA IF NOT EXISTS ANALYTICS;

CREATE WAREHOUSE IF NOT EXISTS MOVIES_ETL_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE MOVIES_ETL_WH;
USE SCHEMA RAW;

SELECT CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE();
```

Expected context: `MOVIES_ETL_DB`, `RAW`, and `MOVIES_ETL_WH`. RAW retains source values, CURATED contains cleaned business data, and ANALYTICS contains a reporting summary.

`IF NOT EXISTS` preserves existing definitions. Use dedicated lab names and verify existing objects if you are reusing them. Stage queries, loads, and transformations use compute credits.

Source: [Working with warehouses](https://docs.snowflake.com/en/user-guide/warehouses-tasks).

## 4. Create the CSV file format and stage

```sql
CREATE FILE FORMAT IF NOT EXISTS MOVIES_ETL_DB.RAW.MOVIELENS_CSV_FORMAT
  TYPE = CSV
  FIELD_DELIMITER = ','
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  EMPTY_FIELD_AS_NULL = TRUE
  ERROR_ON_COLUMN_COUNT_MISMATCH = TRUE
  ENCODING = 'UTF8';

CREATE STAGE IF NOT EXISTS MOVIES_ETL_DB.RAW.MOVIES_STAGE
  FILE_FORMAT = (FORMAT_NAME = 'MOVIES_ETL_DB.RAW.MOVIELENS_CSV_FORMAT');

DESC FILE FORMAT MOVIES_ETL_DB.RAW.MOVIELENS_CSV_FORMAT;
DESC STAGE MOVIES_ETL_DB.RAW.MOVIES_STAGE;
```

The format describes how to read the CSV. It skips the header and treats commas within quoted movie titles as part of one title.

This is a **named internal stage** because its definition has no external storage URL. File bytes are held in Snowflake-managed cloud storage, independently of warehouse compute. `movielens_001/` will be a file-name prefix inside the stage, not a local computer directory.

Sources: [CREATE FILE FORMAT](https://docs.snowflake.com/en/sql-reference/sql/create-file-format), [CREATE STAGE](https://docs.snowflake.com/en/sql-reference/sql/create-stage).

## 5. Upload movies.csv — choose one method

### Method A: Snowsight upload

1. Sign in to Snowsight using the authorized training role.
2. Select **Ingestion → Add Data → Load files into a Stage**.
3. Choose the `C:\data\movielens\ml-latest-small\movies\movies.csv` from your computer.
4. Select database `MOVIES_ETL_DB`, schema `RAW`, stage `MOVIES_STAGE`.
5. Set the destination path to **`movielens_001`** and upload.
6. Continue with the `LIST` check in step 6.

If labels differ in your account, use the stage's upload action. Confirm the remote path after uploading; all following steps expect `movielens_001/movies.csv`.

Source: [Upload files using Snowsight](https://docs.snowflake.com/en/user-guide/data-load-local-file-system-stage-ui).

### Method B: PUT from a local Snowflake client

Run this through an authenticated local Snowflake CLI, SnowSQL, or supported driver:

```sql
PUT 'file://C:/data/movielens/ml-latest-small/movies/movies.csv'
  @MOVIES_ETL_DB.RAW.MOVIES_STAGE/movielens_001/
  AUTO_COMPRESS = FALSE
  OVERWRITE = FALSE;
```

A browser SQL worksheet cannot access your computer's file path through `PUT`. `AUTO_COMPRESS = FALSE` keeps the staged filename `movies.csv`. Repeating an unchanged upload may report `SKIPPED`. Use only one upload method for this batch.

Source: [PUT](https://docs.snowflake.com/en/sql-reference/sql/put).

## 6. List and preview the uploaded movie records

```sql
LIST @MOVIES_ETL_DB.RAW.MOVIES_STAGE/movielens_001/;

SELECT t.$1 AS movie_id_text, t.$2 AS title_text, t.$3 AS genres_text,
       METADATA$FILENAME AS source_file,
       METADATA$FILE_ROW_NUMBER AS source_row
FROM @MOVIES_ETL_DB.RAW.MOVIES_STAGE/movielens_001/
  (FILE_FORMAT => 'MOVIES_ETL_DB.RAW.MOVIELENS_CSV_FORMAT',
   PATTERN => '.*movies[.]csv$') t
ORDER BY source_row
LIMIT 10;
```

Confirm the three fields match the local CSV. Quoted titles containing commas must remain one field, and the header must be skipped. `genres_text` still contains its pipe-separated list.

The following commands expect exactly `movielens_001/movies.csv`. If the upload produces a `.gz` suffix or a different prefix, update every subsequent file reference consistently.

Source: [Query staged files](https://docs.snowflake.com/en/user-guide/querying-stage).

## 7. Create the MovieLens raw table

```sql
CREATE TABLE IF NOT EXISTS MOVIES_ETL_DB.RAW.MOVIELENS_MOVIES_RAW (
  movie_id_text VARCHAR,
  title_text VARCHAR,
  genres_text VARCHAR,
  source_file VARCHAR,
  source_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
```

The three source fields remain strings so invalid IDs can be investigated. File metadata provides lineage. The stage retains the original file while RAW stores parsed source values.

This new table name avoids reusing the previous five-column `MOVIES_RAW` definition.

## 8. Load the file into the raw table

```sql
COPY INTO MOVIES_ETL_DB.RAW.MOVIELENS_MOVIES_RAW
  (movie_id_text, title_text, genres_text, source_file, source_row)
FROM (
  SELECT t.$1, t.$2, t.$3,
         METADATA$FILENAME, METADATA$FILE_ROW_NUMBER
  FROM @MOVIES_ETL_DB.RAW.MOVIES_STAGE/movielens_001/ t
)
FILES = ('movies.csv')
FILE_FORMAT = (FORMAT_NAME = 'MOVIES_ETL_DB.RAW.MOVIELENS_CSV_FORMAT')
ON_ERROR = 'ABORT_STATEMENT'
PURGE = FALSE;

SELECT COUNT(*) AS raw_rows FROM MOVIES_ETL_DB.RAW.MOVIELENS_MOVIES_RAW;
SELECT * FROM MOVIES_ETL_DB.RAW.MOVIELENS_MOVIES_RAW
ORDER BY source_file, source_row LIMIT 10;
```

On the first successful load into an empty table, the raw count should equal the local `Import-Csv` record count. Inspect the COPY status and errors before continuing. The header is excluded; the source file remains staged.

Source: [COPY INTO table](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table).

## 9. Define the transformation rules

| Field | Rule |
|---|---|
| Movie ID | Require a positive whole number |
| Title | Trim spaces and reject a blank title |
| Release year | Extract a four-digit year only from a final `(YYYY)` suffix |
| Genres | Preserve the source list; split on `|` in a separate bridge table |
| Missing genres | Map blank values to `(no genres listed)` |

Preserve the complete title as `title`; also produce `title_without_year`. If no final year is present, keep the movie and leave `release_year` NULL. Do not reject a valid movie solely because its title lacks a year.

Deduplicate by movie ID after validation. For this single-file lab, the earliest source row wins. This is not a latest-update policy for future batches.

## 10. Classify and clean the raw records

```sql
CREATE OR REPLACE VIEW MOVIES_ETL_DB.CURATED.MOVIELENS_CLASSIFIED AS
WITH cleaned AS (
  SELECT movie_id_text, title_text, genres_text,
         TRY_TO_NUMBER(TRIM(movie_id_text), 38, 0) AS movie_id,
         NULLIF(TRIM(title_text), '') AS title,
         COALESCE(NULLIF(TRIM(genres_text), ''), '(no genres listed)') AS genres,
         source_file, source_row, loaded_at
  FROM MOVIES_ETL_DB.RAW.MOVIELENS_MOVIES_RAW
)
SELECT *,
       TRY_TO_NUMBER(REGEXP_SUBSTR(title, '[(]([0-9]{4})[)]$', 1, 1, 'e', 1),
                     4, 0) AS release_year,
       TRIM(REGEXP_REPLACE(title, ' *[(][0-9]{4})[)]$', '')) AS title_without_year,
       CASE
         WHEN movie_id IS NULL OR movie_id <= 0
           OR NOT REGEXP_LIKE(TRIM(movie_id_text), '^[0-9]+$')
           THEN 'INVALID_MOVIE_ID'
         WHEN title IS NULL THEN 'MISSING_TITLE'
         ELSE NULL
       END AS rejection_reason
FROM cleaned;

SELECT movie_id, title, release_year, genres, rejection_reason
FROM MOVIES_ETL_DB.CURATED.MOVIELENS_CLASSIFIED
ORDER BY source_file, source_row LIMIT 20;
```

`TRY_TO_NUMBER` returns NULL for values it cannot convert. The final-year pattern avoids treating a number elsewhere in the title as a release year. NULL rejection reasons indicate accepted rows.

Source: [TRY_TO_NUMBER](https://docs.snowflake.com/en/sql-reference/functions/try_to_decimal).

## 11. Store rejected records

```sql
CREATE OR REPLACE TABLE MOVIES_ETL_DB.RAW.MOVIELENS_REJECTED AS
SELECT movie_id_text, title_text, genres_text,
       source_file, source_row, loaded_at, rejection_reason
FROM MOVIES_ETL_DB.CURATED.MOVIELENS_CLASSIFIED
WHERE rejection_reason IS NOT NULL;

SELECT rejection_reason, COUNT(*) AS rejected_rows
FROM MOVIES_ETL_DB.RAW.MOVIELENS_REJECTED
GROUP BY rejection_reason;
```

The actual file determines the result; a clean MovieLens file may have zero rejects. These are dedicated lab snapshot tables rebuilt from RAW, not an append-only audit history.

## 12. Create the clean movie table

```sql
CREATE OR REPLACE TABLE MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIES AS
SELECT movie_id, title, title_without_year, release_year, genres,
       source_file, source_row
FROM MOVIES_ETL_DB.CURATED.MOVIELENS_CLASSIFIED
WHERE rejection_reason IS NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY movie_id ORDER BY source_file, source_row
) = 1;

SELECT * FROM MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIES ORDER BY movie_id LIMIT 20;
```

There is one row per accepted movie ID. `Toy Story (1995)`, if present, yields `title_without_year = 'Toy Story'` and `release_year = 1995`.

Source: [QUALIFY](https://docs.snowflake.com/en/sql-reference/constructs/qualify).

## 13. Split genres and build the analytics table

```sql
CREATE OR REPLACE TABLE MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIE_GENRES AS
SELECT DISTINCT m.movie_id, TRIM(g.value) AS genre
FROM MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIES m,
LATERAL SPLIT_TO_TABLE(m.genres, '|') g
WHERE TRIM(g.value) <> '';

CREATE OR REPLACE TABLE MOVIES_ETL_DB.ANALYTICS.MOVIELENS_GENRE_SUMMARY AS
SELECT g.genre, COUNT(*) AS movie_count,
       MIN(m.release_year) AS earliest_year,
       MAX(m.release_year) AS latest_year
FROM MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIE_GENRES g
JOIN MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIES m ON m.movie_id = g.movie_id
GROUP BY g.genre;

SELECT * FROM MOVIES_ETL_DB.ANALYTICS.MOVIELENS_GENRE_SUMMARY
ORDER BY movie_count DESC, genre;
```

A movie can have multiple genres. The bridge table stores one row per movie/genre pair, so the sum of genre counts can exceed the number of movies. The `(no genres listed)` marker is retained as its own category. MIN/MAX ignore missing release years.

These tables refresh only when you rerun their creation statements. No rating aggregates are calculated from `movies.csv`.

## 14. Reconcile counts and check uniqueness

```sql
WITH counts AS (
  SELECT
    (SELECT COUNT(*) FROM MOVIES_ETL_DB.RAW.MOVIELENS_MOVIES_RAW) AS raw_rows,
    (SELECT COUNT(*) FROM MOVIES_ETL_DB.RAW.MOVIELENS_REJECTED) AS rejected_rows,
    (SELECT COUNT(*) FROM MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIES) AS curated_rows,
    (SELECT COUNT(*) FROM MOVIES_ETL_DB.CURATED.MOVIELENS_CLASSIFIED
       WHERE rejection_reason IS NULL) AS valid_source_rows
)
SELECT *, valid_source_rows - curated_rows AS duplicates_removed,
       raw_rows = rejected_rows + valid_source_rows AS reconciled
FROM counts;

SELECT movie_id, COUNT(*) AS copies
FROM MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIES
GROUP BY movie_id HAVING COUNT(*) > 1;

SELECT COUNT(*) AS movies,
       COUNT_IF(release_year IS NULL) AS movies_without_year
FROM MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIES;
```

Expected: `RECONCILED = TRUE`, zero rows from the duplicate query, and raw rows equal to the local CSV record count for the first batch. The accounting identity is **raw = rejected + curated + duplicates removed**. Missing years are reported, not rejected. There are no hard-coded sample counts.

## 15. Query the finished movie data

```sql
SELECT m.movie_id, m.title, m.release_year
FROM MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIES m
JOIN MOVIES_ETL_DB.CURATED.MOVIELENS_MOVIE_GENRES g ON m.movie_id = g.movie_id
WHERE g.genre = 'Comedy'
ORDER BY m.release_year DESC NULLS LAST, m.movie_id
LIMIT 20;

SELECT genre, movie_count
FROM MOVIES_ETL_DB.ANALYTICS.MOVIELENS_GENRE_SUMMARY
ORDER BY movie_count DESC, genre;
```

Genre labels retain their MovieLens case. These queries use cleaned tables rather than repeatedly parsing the source file.

## 16. Rerun the same batch correctly

For this one-file exercise:

1. Keep the raw table and stage definitions; `IF NOT EXISTS` does not reset them.
2. If the unchanged file is already staged, skip uploading it again.
3. Re-run `COPY INTO` and inspect its status. Snowflake normally skips previously loaded unchanged files within its load-history rules.
4. Re-run steps 10–14 to rebuild the classification, rejects, curated table, and summary.

Do not add `FORCE = TRUE` casually: it can reload the file into RAW and change the counts. File-load history is not a permanent business-key deduplication system. Dropping/recreating RAW changes the load-history context.

Source: [COPY INTO load behavior](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table).

The derived tables are deliberately replaced on each run, so they do not accumulate duplicate inserts. They are dedicated lab outputs. This multi-step notebook is not one atomic production transaction; a failed step should be fixed and downstream steps rerun before consumers use the new snapshot.

For future batches, use immutable batch paths, a batch audit record, explicit correction/version rules, and orchestration. Those are extensions to this simple manual workflow.

## 17. Troubleshooting

| Problem | Check |
|---|---|
| Local file missing | Confirm `C:\data\movielens\ml-latest-small\movies\movies.csv` exists |
| Wrong column count | Header must be `movieId,title,genres` |
| PUT fails in browser | Use Snowsight upload or a local Snowflake client |
| No staged records | LIST the stage and verify `movielens_001/movies.csv` |
| Quoted title splits | Verify the CSV quote setting |
| Old five-column objects exist | Use the new MOVIELENS-prefixed format and tables |
| Raw count is unexpected | Compare local record count and inspect COPY status/history |
| Year is NULL | The title may not have a final four-digit year; inspect it |
| Genre counts exceed movie count | A movie can belong to several genres |
| Summary is stale | Rebuild the bridge and summary after rebuilding movies |

Restore worksheet context if necessary:

```sql
USE DATABASE MOVIES_ETL_DB;
USE SCHEMA RAW;
USE WAREHOUSE MOVIES_ETL_WH;
```

If the stage already existed with the old default format, explicit `FILE_FORMAT` settings in this notebook still select the MovieLens parser. Inspect existing definitions before reusing a lab environment.

## 18. Finish the lab

```sql
ALTER WAREHOUSE MOVIES_ETL_WH SUSPEND;
```

The lab is complete when the requested CSV is staged, raw rows reconcile to the input, accepted movie IDs are unique, rejected records are inspectable, and genre analytics query successfully.

Suspending the warehouse keeps files and tables available. Storage charges are separate from compute. If the warehouse is already suspended, Snowflake may report that state.

To process another batch, use an immutable batch path and decide how updated movie IDs should replace earlier versions before extending the one-file deduplication rule.